# Task 3: Cross-Country Comparison & Climate Vulnerability Ranking
Objective: Synthesize the cleaned datasets from all five countries to identify relative climate vulnerability and produce a data-driven country ranking to inform Ethiopia's COP32 position paper.

## 1. Setup and Data Loading
We load each country's cleaned CSV and concatenate them into a single DataFrame for comparative analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

In [ ]:
files = [
    'data/ethiopia_clean.csv',
    'data/kenya_clean.csv',
    'data/nigeria_clean.csv',
    'data/sudan_clean.csv',
    'data/tanzania_clean.csv'
]

all_dfs = []
for f in files:
    df = pd.read_csv(f)
    df['Date'] = pd.to_datetime(df['Date'])
    all_dfs.append(df)

df_all = pd.concat(all_dfs, ignore_index=True)
df_all.head()

**Action:** Loaded cleaned datasets for Ethiopia, Kenya, Nigeria, Sudan, and Tanzania, ensuring 'Date' column is in datetime format and concatenated them into a single master DataFrame.

## 2. Temperature Trend Comparison
We compare the monthly average temperature (T2M) trends across all five countries from 2015 to 2026.

In [ ]:
df_all['MonthYear'] = df_all['Date'].dt.to_period('M')
monthly_temp = df_all.groupby(['Country', 'MonthYear'])['T2M'].mean().reset_index()
monthly_temp['MonthYear'] = monthly_temp['MonthYear'].dt.to_timestamp()

plt.figure(figsize=(15, 7))
for country in monthly_temp['Country'].unique():
    subset = monthly_temp[monthly_temp['Country'] == country]
    plt.plot(subset['MonthYear'], subset['T2M'], label=country, linewidth=1.5, alpha=0.8)

plt.title("Cross-Country Comparison: Monthly Average Temperature (2015-2026)", fontsize=16)
plt.xlabel("Year")
plt.ylabel("Average Temperature (T2M, °C)")
plt.legend(title="Country", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Summary — Temperature Trend Analysis:**
The multi-country line chart reveals distinct thermal regimes. Sudan consistently exhibits the highest average temperatures, while Ethiopia shows a cooler, more varied profile likely due to its high-altitude topography. The seasonal synchronicity across East Africa (Ethiopia, Kenya, Tanzania) is visible through similar peak/trough patterns.

## 3. Statistical Comparison of Temperature (T2M)
A quantitative summary of thermal conditions across the five regions.

In [ ]:
temp_stats = df_all.groupby('Country')['T2M'].agg(['mean', 'median', 'std']).sort_values(by='mean', ascending=False)
temp_stats.style.background_gradient(cmap='YlOrRd')

**Summary Statistics Interpretation:**
The statistical comparison ranks Sudan as the warmest region with the highest mean temperature. Nigeria follows, exhibiting tropical consistency. Tanzania and Kenya show moderate thermal regimes, while Ethiopia maintains the lowest mean temperature among the group, serving as a 'water tower' with a unique cooling profile.

## 4. Precipitation Comparison
We compare the average annual precipitation across the five countries to identify regions at higher risk of drought.

In [ ]:
precip_stats = df_all.groupby('Country')['PRECTOTCORR'].mean().sort_values(ascending=True).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=precip_stats, x='Country', y='PRECTOTCORR', palette='Blues_d')
plt.title("Average Daily Precipitation Comparison (mm/day)", fontsize=16)
plt.ylabel("Mean Precipitation (mm/day)")
plt.show()

**Summary — Precipitation Analysis:**
Sudan exhibits the lowest average precipitation, highlighting its vulnerability to arid conditions. Nigeria and Tanzania show significantly higher rainfall totals, characteristic of their tropical and coastal climates respectively. Ethiopia and Kenya occupy the middle ground, with bimodal rainfall patterns that are critical for their agricultural sectors.

## 5. Climate Vulnerability Index (CVI) Calculation
The CVI is calculated based on three normalized factors:
1. **Temperature Stress**: Mean Temperature (T2M)
2. **Thermal Instability**: Temperature Standard Deviation (Variability)
3. **Water Stress**: Inverse of Mean Precipitation (1 - Normalized Precip)

A higher CVI score indicates higher climate vulnerability.

In [ ]:
vulnerability_metrics = df_all.groupby('Country').agg({
    'T2M': ['mean', 'std'],
    'PRECTOTCORR': 'mean'
}).reset_index()

vulnerability_metrics.columns = ['Country', 'Mean_Temp', 'Temp_Std', 'Mean_Precip']

def normalize(series):
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

vulnerability_metrics['Norm_Temp'] = normalize(vulnerability_metrics['Mean_Temp'])
vulnerability_metrics['Norm_Var'] = normalize(vulnerability_metrics['Temp_Std'])
vulnerability_metrics['Norm_Precip'] = normalize(vulnerability_metrics['Mean_Precip'])


vulnerability_metrics['CVI'] = (
    vulnerability_metrics['Norm_Temp'] + 
    vulnerability_metrics['Norm_Var'] + 
    (1 - vulnerability_metrics['Norm_Precip'])
) / 3

vulnerability_metrics = vulnerability_metrics.sort_values(by='CVI', ascending=False)
print(vulnerability_metrics[['Country', 'Mean_Temp', 'Mean_Precip', 'CVI']])

**Summary — Climate Vulnerability Ranking:**
Based on the synthesized data, **Sudan** emerges as the most vulnerable country due to its extreme heat and low rainfall. **Ethiopia** shows high vulnerability linked to temperature variability, despite its lower mean temperatures. **Nigeria** and **Tanzania** appear relatively more resilient in this specific index due to higher water availability, although they face different risks like coastal flooding not captured here.

## 6. Final Ranking Visualization
Visualizing the Climate Vulnerability Index (CVI) across the research regions.

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=vulnerability_metrics, x='Country', y='CVI', palette='Reds_r')
plt.title("Climate Vulnerability Index (CVI) Ranking for COP32 Policy Framework", fontsize=16)
plt.ylabel("CVI Score (Normalized)")
plt.ylim(0, 1)
plt.show()